# Predicting late deliveries
Goal: predict `late_delivery` at dispatch time, for consignments later than the data.

Design (per checklist.md): rows are ordered in time and each consignment appears once, so the test set is the **latest 20%** of rows by `dispatched_on`; cross-validation inside training uses expanding-window splits (earlier train, later validate).
`claim_paid_eur` is settled after delivery (a consequence of lateness), so it is excluded; `consignment` is an identifier and is dropped.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
df=pd.read_csv('data.csv',parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
print(df.shape, df.consignment.is_unique, df.isna().sum().sum())
df['hour']=df.dispatched_on.dt.hour; df['dow']=df.dispatched_on.dt.dayofweek

## 1. Hold out the test set (latest 20%) before any exploration of outcomes

In [ ]:
cut=int(len(df)*0.8)
train,test=df.iloc[:cut].copy(),df.iloc[cut:].copy()
print(train.dispatched_on.max(), test.dispatched_on.min())
FEATS=['carrier','distance_km','weight_kg','parcels','hour','dow']
X,y=train[FEATS],train.late_delivery

## 2. Exploration (training rows only)

In [ ]:
print(train.groupby(train.dispatched_on.dt.to_period('M')).late_delivery.agg(['mean','size']).T.round(3).to_string())
print(train.groupby('carrier').late_delivery.mean())

The late rate drifts strongly over time, so the future base rate may differ from training. No calendar/trend feature is used because it cannot extrapolate; hour and weekday are kept.

## 3. Candidates and selection protocol (declared before scoring)
Candidates: (A) trivial baseline predicting the training prior; (B) logistic regression; (C) logistic with C=0.1; (D) shallow gradient boosting; (E) deeper gradient boosting with min leaf 100.
Selection metric: mean ROC AUC over expanding-window CV folds within the training rows. Decision threshold: chosen on pooled CV out-of-fold predictions to maximise F1. Preprocessing lives inside the Pipeline so it is fit per fold.

In [ ]:
num=['distance_km','weight_kg','parcels','hour','dow']
def pre(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),['carrier']),('n',StandardScaler() if scale else 'passthrough',num)])
def mk(name):
    return {'A_baseline':Pipeline([('p',pre(False)),('m',DummyClassifier(strategy='prior'))]),
    'B_logit':Pipeline([('p',pre(True)),('m',LogisticRegression(max_iter=1000))]),
    'C_logit_C0.1':Pipeline([('p',pre(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
    'D_hgb_shallow':Pipeline([('p',pre(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=150,random_state=0))]),
    'E_hgb_leaf':Pipeline([('p',pre(False)),('m',HistGradientBoostingClassifier(max_depth=5,min_samples_leaf=100,learning_rate=0.05,max_iter=150,random_state=0))])}[name]
names=['A_baseline','B_logit','C_logit_C0.1','D_hgb_shallow','E_hgb_leaf']
cv=TimeSeriesSplit(n_splits=5)
oof={}; rows=[]
for n in names:
    aucs=[]; idx=[]; pr=[]
    for tr,va in cv.split(X):
        m=mk(n).fit(X.iloc[tr],y.iloc[tr]); p=m.predict_proba(X.iloc[va])[:,1]
        aucs.append(roc_auc_score(y.iloc[va],p)); idx.append(va); pr.append(p)
    oof[n]=(np.concatenate(idx),np.concatenate(pr))
    rows.append((n,np.mean(aucs),np.std(aucs)))
cvres=pd.DataFrame(rows,columns=['model','cv_auc','sd']).set_index('model'); cvres

In [ ]:
best=cvres.cv_auc.idxmax(); print('chosen:',best)
idx,p=oof[best]; yy=y.iloc[idx].values
ths=np.linspace(0.05,0.8,76); f1s=[f1_score(yy,p>=t) for t in ths]
thr=float(ths[int(np.argmax(f1s))]); print('threshold',thr,'cv F1',max(f1s))

## 4. Choice (fixed before touching the test set)
The model and threshold printed above are final. The model is refit on all training rows and scored once on the test set.

In [ ]:
final=mk(best).fit(X,y)
pt=final.predict_proba(test[FEATS])[:,1]; yt=test.late_delivery
base=mk('A_baseline').fit(X,y).predict_proba(test[FEATS])[:,1]
print('test base rate',yt.mean(),'train base rate',y.mean())
print(f'TEST ROC AUC {roc_auc_score(yt,pt):.3f}  PR AUC {average_precision_score(yt,pt):.3f}  F1@thr {f1_score(yt,pt>=thr):.3f}  Brier {brier_score_loss(yt,pt):.3f}')
print(f'Baseline Brier {brier_score_loss(yt,base):.3f}, PR AUC of a no-skill model (= prevalence) {yt.mean():.3f}')

## 5. Conclusion
See the printed test metrics above: that is the one-shot headline. Caveats: the late rate drifts heavily over time (very low in the final months), so a threshold tuned on earlier periods and the calibration may not transfer; and the available features (carrier, distance, weight, parcels, dispatch time) may carry limited signal. Judge AUC against 0.5 and PR AUC against the test prevalence.